# 03 — Reproduce the published NOWRITE result

**Stage:** Gautam's step 2, and the proposal's **Week 6 milestone**: *"Reproducing the
published 38–42% changed-answer rate under NOWRITE on the 3B checkpoints is the cleanest
available evidence that our instrumented fork behaves like the real system … we treat
failure as a blocker."*

Nothing in the 18 Aug pilot measures a task score. `AHN_algoverse.ipynb` cell 16 loops
over 20 RULER examples and reports `‖o_t‖` norms — a plumbing statistic, not an answer.
This notebook produces the first real numbers: mean F1, ΔF1, and **answer-change rate**
under AHN vs NOWRITE, on LongBench-E HotpotQA at the proposal's settings.

Target from the concurrent write-attrition study: **38–42% of answers change** while
mean F1 moves only **0.4–2.3 points**. Both halves matter — a large F1 swing would be as
much a red flag as no answer changes at all.

Cost: 60 examples × 2 conditions, ~4–32K tokens each. Budget 2–4 GPU-hours at 3B.


In [1]:
# --- GPU slice: CHANGE THIS EVERY RUN --------------------------------------------
# The H100s are MIG-partitioned: a process gets one ~20 GB slice, not a whole card,
# and the slice UUIDs are regenerated every time the box is recycled (~48h). A bare
# index ("0", "3") does NOT select a MIG slice -- it silently lands somewhere else.
#   nvidia-smi -L    list the slices
#   nvidia-smi       see which are actually idle (four of us share this box)
# Exporting in the shell does not reach the Jupyter kernel, and CUDA reads this once
# at init, so it must be set here -- before anything imports torch.
import os
MIG_UUID = ""   # <- paste, e.g. "MIG-802c3ecc-8c66-53d4-9fb5-60712ea8f619"
if MIG_UUID:
    os.environ["CUDA_VISIBLE_DEVICES"] = MIG_UUID
elif not os.environ.get("CUDA_VISIBLE_DEVICES"):
    print("! MIG_UUID empty and CUDA_VISIBLE_DEVICES unset -- this kernel lands on "
          "whatever slice it defaults to, possibly one a teammate is using. "
          "Run `nvidia-smi -L`, pick an idle slice, paste its UUID above.")

# --- bootstrap -------------------------------------------------------------------
# `ahn_interp.py` lives at the repo root; this walks up the tree to find it.
# Run this notebook from inside the clone -- nothing needs uploading.
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Run this notebook from inside the repo "
               "clone -- `git clone` it on the box rather than copying notebooks "
               "around; the bootstrap searches four levels up from the cwd.")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-23fd/Interpretability-study-of-Artificial-Hippocampus-Networks
working directory pinned to /home/jupyter-dphs-23fd/Interpretability-study-of-Artificial-Hippocampus-Networks


In [2]:
# --- experiment configuration ----------------------------------------------------
# Select exactly one config. Each recurrent cell writes to its own run directory.
RUN_CONFIG = "configs/run_3b_gdn.json"  # run_3b_gdn | run_3b_dn | run_3b_m2
ALLOW_OVERWRITE_COMPLETED = False          # set True only for a deliberate rerun
OUTPUT_NAME = "03_nowrite_reproduction.json"
EXPECTED_N = 60

with open(RUN_CONFIG) as f:
    run_cfg = json.load(f)
CFG = dict(
    model_path      = ai.resolve_ckpt(run_cfg["ckpt_name"]),
    cell            = run_cfg["cell"],
    scale           = run_cfg["scale"],
    sliding_window  = run_cfg["sliding_window"],
    num_attn_sinks  = run_cfg["num_attn_sinks"],
    attn_impl       = run_cfg["attn_impl"],
    dtype           = run_cfg["dtype"],
    results_dir     = os.path.join("results", run_cfg["run_name"]),
)
ai.set_results_dir(CFG["results_dir"])
OUTPUT_PATH = os.path.join(CFG["results_dir"], OUTPUT_NAME)

if os.path.exists(OUTPUT_PATH) and not ALLOW_OVERWRITE_COMPLETED:
    with open(OUTPUT_PATH) as f:
        prior = json.load(f)
    prior_rows = prior.get("per_example", [])
    if len(prior_rows) >= EXPECTED_N:
        raise FileExistsError(
            f"Completed run already exists at {OUTPUT_PATH} ({len(prior_rows)} rows). "
            "Set ALLOW_OVERWRITE_COMPLETED=True only after deciding to replace it."
        )

print(json.dumps(CFG, indent=2))
print("output:", OUTPUT_PATH)


{
  "model_path": "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
import torch, time, numpy as np

bundle = ai.load_ahn_model(
    CFG["model_path"],
    dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"],
    num_attn_sinks=CFG["num_attn_sinks"],
)

tok = bundle.tokenizer
print("loaded:", bundle.ahn_impl, "| window", bundle.sliding_window,
      "| sinks", bundle.num_attn_sinks)


/home/jupyter-dphs-23fd/ahn-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.60it/s]


loaded: GatedDeltaNet | window 8064 | sinks 128


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.78it/s]

loaded: GatedDeltaNet | window 8064 | sinks 128


In [ ]:
from datasets import load_dataset

N_EXAMPLES = 60          # matches the concurrent study's cohort size
MAX_INPUT   = 32000
DATASET     = "hotpotqa"

ds = load_dataset("THUDM/LongBench", f"{DATASET}_e", split="test")
prompt_tmpl = (
    "Answer the question based on the given passages. Only give me the answer and do "
    "not output any other words.\n\nThe following are given passages.\n{context}\n\n"
    "Answer the question based on the given passages. Only give me the answer and do "
    "not output any other words.\n\nQuestion: {input}\nAnswer:"
)

rows = []
for ex in ds:
    p = prompt_tmpl.format(context=ex["context"], input=ex["input"])
    n = len(tok.encode(p))
    if n > MAX_INPUT or n <= bundle.sliding_window + bundle.num_attn_sinks:
        continue          # AHN must actually be active, or the comparison is empty
    rows.append({"prompt": p, "answers": ex["answers"], "n_tokens": n,
                 "length_bucket": ex.get("length", None), "_id": ex.get("_id", len(rows))})

# length-stratified sample: shortest / median / longest thirds (proposal, RQ1 analysis)
rows.sort(key=lambda r: r["n_tokens"])
third = max(1, len(rows) // 3)
buckets = {"short": rows[:third], "mid": rows[third:2*third], "long": rows[2*third:]}
rng = np.random.default_rng(ai.SEED)
cohort = []
per = N_EXAMPLES // 3
for name, b in buckets.items():
    idx = rng.choice(len(b), size=min(per, len(b)), replace=False)
    for i in idx:
        r = dict(b[int(i)]); r["stratum"] = name; cohort.append(r)
print(f"eligible {len(rows)} -> cohort {len(cohort)}")
print({k: sum(1 for c in cohort if c["stratum"] == k) for k in buckets})
print("token range:", min(c["n_tokens"] for c in cohort), "-", max(c["n_tokens"] for c in cohort))


In [ ]:
@torch.no_grad()
def predict(prompt, nowrite=False, max_new_tokens=32):
    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    handles = []
    if nowrite:
        def z(m, i, o):
            return (torch.zeros_like(o[0]),) + o[1:] if isinstance(o, tuple) else torch.zeros_like(o)
        for L in bundle.ahn_layers:
            handles.append(bundle.model.model.layers[L].ahn.register_forward_hook(z))
    try:
        out = bundle.model.generate(
            **ins, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tok.eos_token_id, return_dict_in_generate=True,
            output_scores=True,
        )
    finally:
        for h in handles: h.remove()
    answer = tok.decode(
        out.sequences[0, ins["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    # scores[0] is the next-token distribution at the prompt/compression boundary.
    boundary_logits = out.scores[0][0].detach().float().cpu()
    return answer, boundary_logits

def first_line(text):
    return text.split("\n", 1)[0].strip()

def js_divergence(logits_p, logits_q):
    p = torch.softmax(logits_p, dim=-1)
    q = torch.softmax(logits_q, dim=-1)
    m = 0.5 * (p + q)
    eps = 1e-12
    return float(0.5 * (
        (p * (p.clamp_min(eps).log() - m.clamp_min(eps).log())).sum()
        + (q * (q.clamp_min(eps).log() - m.clamp_min(eps).log())).sum()
    ))

results, t0 = [], time.time()
for i, c in enumerate(cohort):
    a_on, logits_on   = predict(c["prompt"], nowrite=False)
    a_off, logits_off = predict(c["prompt"], nowrite=True)
    f1_on  = max(ai.qa_f1_score(a_on,  g) for g in c["answers"])
    f1_off = max(ai.qa_f1_score(a_off, g) for g in c["answers"])
    a_on_fl, a_off_fl = first_line(a_on), first_line(a_off)
    f1_on_fl  = max(ai.qa_f1_score(a_on_fl,  g) for g in c["answers"])
    f1_off_fl = max(ai.qa_f1_score(a_off_fl, g) for g in c["answers"])
    results.append({
        "id": c["_id"], "stratum": c["stratum"], "n_tokens": c["n_tokens"],
        "answer_ahn": a_on, "answer_nowrite": a_off,
        "f1_ahn": f1_on, "f1_nowrite": f1_off, "delta_f1": f1_on - f1_off,
        "answer_changed": ai.normalize_answer(a_on) != ai.normalize_answer(a_off),
        "answer_ahn_fl": a_on_fl, "answer_nowrite_fl": a_off_fl,
        "f1_ahn_fl": f1_on_fl, "f1_nowrite_fl": f1_off_fl,
        "delta_f1_fl": f1_on_fl - f1_off_fl,
        "answer_changed_fl": (
            ai.normalize_answer(a_on_fl) != ai.normalize_answer(a_off_fl)
        ),
        "boundary_js": js_divergence(logits_on, logits_off),
        "gold": c["answers"],
    })
    if (i + 1) % 5 == 0:
        cc = np.mean([r["answer_changed_fl"] for r in results])
        print(f"[{i+1:3d}/{len(cohort)}] changed={cc:.1%} "
              f"F1 {np.mean([r['f1_ahn_fl'] for r in results]):.3f}/"
              f"{np.mean([r['f1_nowrite_fl'] for r in results]):.3f} "
              f"({(time.time()-t0)/60:.1f} min)")
    del logits_on, logits_off
    ai.free_cuda()
print(f"\ndone in {(time.time()-t0)/60:.1f} min")


In [ ]:
def build_rq1_summary(saved_rows):
    """Build every headline RQ1 statistic from saved first-line fields only."""
    required = {"f1_ahn_fl", "f1_nowrite_fl", "delta_f1_fl",
                "answer_changed_fl", "boundary_js"}
    missing = [(r.get("id"), sorted(required - r.keys())) for r in saved_rows
               if not required.issubset(r)]
    assert not missing, f"per-example rows lack required fields: {missing[:3]}"

    change_rate = float(np.mean([r["answer_changed_fl"] for r in saved_rows]))
    f1_on = ai.bootstrap_ci([r["f1_ahn_fl"] for r in saved_rows])
    f1_off = ai.bootstrap_ci([r["f1_nowrite_fl"] for r in saved_rows])
    d_f1 = ai.bootstrap_ci([r["delta_f1_fl"] for r in saved_rows])
    boundary_js = ai.bootstrap_ci([r["boundary_js"] for r in saved_rows])
    summary = {
        "n": len(saved_rows),
        "answer_change_rate": change_rate,
        "answer_change_rate_ci": ai.bootstrap_ci(
            [float(r["answer_changed_fl"]) for r in saved_rows]
        )[1:],
        "mean_f1_ahn": f1_on, "mean_f1_nowrite": f1_off, "delta_f1": d_f1,
        "delta_f1_points": d_f1[0] * 100,
        "boundary_js": boundary_js,
        "published_change_rate_range": [0.38, 0.42],
        "published_f1_shift_points": [0.4, 2.3],
        "per_stratum": {
            s: {
                "n": sum(r["stratum"] == s for r in saved_rows),
                "change_rate": float(np.mean([
                    r["answer_changed_fl"] for r in saved_rows if r["stratum"] == s
                ])),
                "delta_f1": ai.bootstrap_ci([
                    r["delta_f1_fl"] for r in saved_rows if r["stratum"] == s
                ]),
            } for s in ("short", "mid", "long")
        },
        "metric_note": ("Headline answer-change and F1 statistics are regenerated "
            "exclusively from the saved first-line per-example fields; full-generation "
            "fields are retained for audit only."),
        "cfg": CFG, "dataset": DATASET, "max_input": MAX_INPUT,
    }
    summary["reproduction_ok"] = bool(
        0.30 <= change_rate <= 0.50 and abs(d_f1[0] * 100) <= 5.0
    )
    return summary

repro = build_rq1_summary(results)
ai.save_json({"summary": repro, "per_example": results}, OUTPUT_NAME)
print(json.dumps(repro, indent=2, default=str))
print("\nWEEK-6 MILESTONE:", "PASS" if repro["reproduction_ok"] else "FAIL — blocker, tell Gautam")


In [ ]:
# Reload the artifact and regenerate the summary from its saved first-line fields.
# This is both a consistency check and a repair path for any interrupted analysis cell.
with open(OUTPUT_PATH) as f:
    saved = json.load(f)
saved["summary"] = build_rq1_summary(saved["per_example"])
assert len(saved["per_example"]) == EXPECTED_N
assert all("boundary_js" in r for r in saved["per_example"])
ai.save_json(saved, OUTPUT_NAME)
print(f"validated {len(saved['per_example'])} rows -> {OUTPUT_PATH}")
print("WEEK-6 MILESTONE:",
      "PASS" if saved["summary"]["reproduction_ok"] else "FAIL — blocker, tell Gautam")

### Reading the result

- **Change rate 38–42%, |ΔF1| under ~2.5 points** → the instrumented fork behaves like
  the real system. This is the sentence that unblocks everything else, and it belongs in
  the next mentor update verbatim.
- **Change rate far below 38%** → AHN is barely active. Check `n_tokens` vs
  `sliding_window + num_attn_sinks`; the cohort filter above should have prevented this.
- **Change rate near 38% but ΔF1 large** → decoding config drift (sampling on, wrong
  chat template, wrong `max_new_tokens`). Compare against the upstream `eval/longbench`
  settings.

`per_example` in the saved JSON is the RQ3 join key: `delta_f1` per example is exactly
the outcome variable Table 8 correlates retention against. Keep it.
